# Linear Regression: From Intuition to Implementation

A comprehensive guide to understanding, implementing, and evaluating linear regression models — from the basic equation to real-world deployment considerations.

**What you will learn:**
- The core intuition behind regression and when to apply it
- Mathematical foundations of Ordinary Least Squares
- How to implement linear regression from scratch using matrix operations
- Professional workflows using scikit-learn
- Diagnostic tools for model validation
- Regularization techniques to combat overfitting
- Real-world modeling with the California housing dataset
- How to diagnose bias-variance tradeoff using learning curves

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 1. What is Regression? When to Use It

**Regression** is a supervised learning technique used to model the relationship between a dependent variable (target) and one or more independent variables (features). Unlike classification (predicting discrete labels), regression predicts **continuous numeric values**.

### Common use cases:
- **Predicting house prices** based on square footage, bedrooms, location
- **Forecasting sales** from advertising spend across channels
- **Estimating temperature** given time of day, humidity, pressure
- **Analyzing factors affecting** employee performance, patient recovery time, etc.

### When to use linear regression:
- The relationship between features and target is approximately linear
- You need interpretable coefficients (understanding _how_ each feature affects the prediction)
- You have a relatively clean dataset with limited multicollinearity
- As a strong baseline before trying more complex models

In [ ]:
# Quick visual: what regression looks like
X_demo = np.linspace(0, 10, 50)
y_demo = 2.5 * X_demo + 5 + np.random.normal(0, 2, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_demo, y_demo, alpha=0.7, edgecolors='k')
axes[0].set_title('Regression Problem: Predict y from X')
axes[0].set_xlabel('Feature (X)')
axes[0].set_ylabel('Target (y)')

# Show classification analog for contrast
X_clf = np.random.randn(50, 2)
y_clf = (X_clf[:, 0] + X_clf[:, 1] > 0).astype(int)
axes[1].scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, cmap='bwr', edgecolors='k', alpha=0.7)
axes[1].set_title('Classification Problem: Predict Discrete Label')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 2. Simple Linear Regression: $y = mx + c$ Intuition

Simple linear regression models the relationship between **one feature** $x$ and the target $y$:

$$y = \beta_0 + \beta_1 x + \varepsilon$$

Where:
- $\beta_0$ is the **intercept** (value of $y$ when $x = 0$)
- $\beta_1$ is the **slope** (change in $y$ per unit change in $x$)
- $\varepsilon$ is the **error term** (what the model cannot explain)

**Intuition:** We are trying to draw the "best" straight line through our data points. "Best" means the line that minimizes the sum of squared vertical distances between the points and the line.

In [ ]:
# Visualize the intuition: find the line of best fit
x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
y = np.array([2.1, 2.9, 4.2, 5.0, 6.1, 6.8, 8.1, 9.0])

# Fit via numpy polyfit for demo
beta_1, beta_0 = np.polyfit(x, y, 1)
y_pred = beta_0 + beta_1 * x
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: data + line
axes[0].scatter(x, y, s=80, color='steelblue', edgecolors='k', zorder=5)
axes[0].plot(x, y_pred, 'r--', linewidth=2, label=f'y = {beta_0:.2f} + {beta_1:.2f}x')
for i in range(len(x)):
    axes[0].vlines(x[i], y[i], y_pred[i], color='gray', linestyle=':', alpha=0.6)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Line of Best Fit with Residuals')
axes[0].legend(fontsize=11)

# Right: sum of squared residuals
axes[1].bar(range(len(x)), residuals**2, color='coral', alpha=0.7, edgecolor='k')
axes[1].set_xlabel('Data point')
axes[1].set_ylabel('Squared residual')
axes[1].set_title(f'Sum of Squared Residuals: {np.sum(residuals**2):.2f}')
axes[1].axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print(f'Intercept (beta_0): {beta_0:.4f}')
print(f'Slope     (beta_1): {beta_1:.4f}')
print(f'Equation: y = {beta_0:.2f} + {beta_1:.2f} * x')

## 3. Ordinary Least Squares (OLS) Derivation

OLS finds coefficients $\boldsymbol{\beta}$ that minimize the sum of squared residuals:

$$\text{RSS}(\boldsymbol{\beta}) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \sum_{i=1}^{n} (y_i - \mathbf{x}_i^T \boldsymbol{\beta})^2$$

### In matrix form:

$$\text{RSS}(\boldsymbol{\beta}) = (\mathbf{y} - \mathbf{X}\boldsymbol{\beta})^T (\mathbf{y} - \mathbf{X}\boldsymbol{\beta})$$

### Steps (plain English):
1. Write the error as the difference between actual and predicted values
2. Square it (so positive and negative errors don't cancel)
3. Take the derivative with respect to $\boldsymbol{\beta}$
4. Set the derivative to zero (minimum occurs where slope = 0)
5. Solve for $\boldsymbol{\beta}$

### The closed-form solution:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

This is the **Normal Equation**. It gives us the optimal coefficients in one shot — no iterative optimization needed.

## 4. Multiple Linear Regression

When we have $p$ features instead of one, the model extends naturally:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p + \varepsilon$$

Each coefficient $\beta_j$ represents the **average change** in $y$ for a one-unit change in $x_j$, **holding all other features constant** (the *ceteris paribus* interpretation).

In [ ]:
# Generate synthetic data with 3 features
np.random.seed(123)
n = 200
X_multi = np.random.randn(n, 3)
true_coefs = np.array([3.5, -1.2, 0.8])
y_multi = 5.0 + X_multi @ true_coefs + np.random.normal(0, 1.5, n)

# Fit multiple linear regression
X_multi_with_intercept = np.c_[np.ones(n), X_multi]
beta_hat = np.linalg.inv(X_multi_with_intercept.T @ X_multi_with_intercept) @ X_multi_with_intercept.T @ y_multi

print('Estimated coefficients:')
print(f'Intercept (beta_0): {beta_hat[0]:.4f}  (true: 5.0)')
for i in range(3):
    print(f'beta_{i+1}: {beta_hat[i+1]:.4f}  (true: {true_coefs[i]})')

## 5. Assumptions of Linear Regression

For valid inference and reliable predictions, linear regression relies on these assumptions:

| Assumption | Description | How to check |
|---|---|---|
| **Linearity** | The relationship between X and y is linear | Scatter plots, residuals vs fitted |
| **Independence** | Observations are independent of each other | Domain knowledge, Durbin-Watson test |
| **Homoscedasticity** | Constant variance of residuals across all fitted values | Residuals vs fitted plot (no funnel shape) |
| **Normality** | Residuals are normally distributed (for inference) | Q-Q plot, histogram of residuals |
| **No multicollinearity** | Predictors are not highly correlated | Variance Inflation Factor (VIF) |

**Important:** For prediction only, normality and some heteroscedasticity are tolerable. The most critical assumption is **linearity**.

## 6. Implementing from Scratch Using Matrix Operations

In [ ]:
class LinearRegressionScratch:
    """Linear regression implemented from scratch using the Normal Equation."""
    
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = 0.0
    
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        
        if self.fit_intercept:
            X = np.c_[np.ones(X.shape[0]), X]
        
        # Normal Equation: beta = (X^T X)^-1 X^T y
        XtX = X.T @ X
        XtX_inv = np.linalg.inv(XtX)
        beta = XtX_inv @ X.T @ y
        
        if self.fit_intercept:
            self.intercept_ = beta[0, 0]
            self.coef_ = beta[1:, 0]
        else:
            self.coef_ = beta[:, 0]
        
        return self
    
    def predict(self, X):
        X = np.asarray(X)
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - ss_res / ss_tot


# Test against scikit-learn
np.random.seed(42)
X_test = np.random.randn(100, 2)
y_test = 4 + 2.5 * X_test[:, 0] - 1.3 * X_test[:, 1] + np.random.normal(0, 0.5, 100)

scratch_model = LinearRegressionScratch()
scratch_model.fit(X_test, y_test)

sk_model = LinearRegression()
sk_model.fit(X_test, y_test)

print('Our implementation:')
print(f'  Intercept: {scratch_model.intercept_:.4f}')
print(f'  Coefs: {scratch_model.coef_}')
print(f'  R²: {scratch_model.score(X_test, y_test):.4f}')
print()
print('scikit-learn:')
print(f'  Intercept: {sk_model.intercept_:.4f}')
print(f'  Coefs: {sk_model.coef_}')
print(f'  R²: {sk_model.score(X_test, y_test):.4f}')

## 7. Implementation with scikit-learn (LinearRegression)

In [ ]:
# Generate a larger dataset
X, y = make_regression(n_samples=500, n_features=5, noise=15, random_state=42)
feature_names = [f'Feature_{i}' for i in range(5)]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)
y_test_pred = lr.predict(X_test)

print('Coefficients:')
for name, coef in zip(feature_names, lr.coef_):
    print(f'  {name}: {coef:.4f}')
print(f'Intercept: {lr.intercept_:.4f}')
print(f'\nTrain R²: {lr.score(X_train, y_train):.4f}')
print(f'Test  R²: {lr.score(X_test, y_test):.4f}')

## 8. Evaluation Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Average squared error (penalizes large errors heavily) |
| **RMSE** | $\sqrt{\text{MSE}}$ | In same units as y (most interpretable) |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error (robust to outliers) |
| **R²** | $1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$ | Proportion of variance explained by the model |
| **Adjusted R²** | $1 - \frac{(1-R^2)(n-1)}{n-p-1}$ | R² penalized for number of features |
| **MAPE** | $\frac{100\%}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Mean Absolute Percentage Error |

In [ ]:
def adjusted_r2(r2, n, p):
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

n_train, p = X_train.shape

metrics = {
    'R²': r2_score(y_test, y_test_pred),
    'Adjusted R²': adjusted_r2(r2_score(y_test, y_test_pred), n_train, p),
    'MAE': mean_absolute_error(y_test, y_test_pred),
    'MSE': mean_squared_error(y_test, y_test_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
    'MAPE (%)': mape(y_test, y_test_pred)
}

metrics_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
metrics_df['Value'] = metrics_df['Value'].map(lambda v: f'{v:.4f}')
print(metrics_df.to_string(index=False))

## 9. Visualizing Residuals

Residual diagnostics help us verify model assumptions and identify problems.

In [ ]:
residuals = y_test - y_test_pred
std_residuals = residuals / np.std(residuals)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(y_test_pred, residuals, alpha=0.6, edgecolors='k')
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (check linearity & homoscedasticity)')

# 2. Q-Q plot for normality
stats.probplot(residuals, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (check normality)')
axes[0, 1].get_lines()[0].set_markerfacecolor('steelblue')
axes[0, 1].get_lines()[0].set_markeredgecolor('k')
axes[0, 1].get_lines()[0].set_alpha(0.6)

# 3. Scale-Location (sqrt of standardized residuals vs fitted)
axes[1, 0].scatter(y_test_pred, np.sqrt(np.abs(std_residuals)), alpha=0.6, edgecolors='k')
from scipy.interpolate import UnivariateSpline
sorted_idx = np.argsort(y_test_pred)
spline = UnivariateSpline(y_test_pred[sorted_idx], np.sqrt(np.abs(std_residuals[sorted_idx])), s=100)
axes[1, 0].plot(y_test_pred[sorted_idx], spline(y_test_pred[sorted_idx]), 'r-', linewidth=2)
axes[1, 0].set_xlabel('Fitted values')
axes[1, 0].set_ylabel(r'$\sqrt{|\text{Standardized Residuals}|}$')
axes[1, 0].set_title('Scale-Location (check homoscedasticity)')

# 4. Histogram of residuals
axes[1, 1].hist(residuals, bins=25, density=True, alpha=0.6, color='steelblue', edgecolor='k')
from scipy.stats import norm
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 1].plot(x_range, norm.pdf(x_range, residuals.mean(), residuals.std()), 'r-', linewidth=2)
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f'Shapiro-Wilk normality test p-value: {stats.shapiro(residuals[:5000])[1]:.6f}')
print(f'(If p > 0.05, residuals are approximately normal)')

## 10. Polynomial Regression

When the relationship is not linear, we can add polynomial features:

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d + \varepsilon$$

**Caution:** Higher-degree polynomials can severely overfit, especially at the boundaries.

In [ ]:
# Create a non-linear relationship
np.random.seed(42)
X_poly = np.linspace(-3, 3, 100).reshape(-1, 1)
y_poly = 0.5 * X_poly.ravel()**2 + X_poly.ravel() + 2 + np.random.normal(0, 1, 100)

X_poly_train, X_poly_test, y_poly_train, y_poly_test = train_test_split(
    X_poly, y_poly, test_size=0.3, random_state=42
)

degrees = [1, 2, 5, 15]
colors = ['gray', 'green', 'orange', 'red']

plt.figure(figsize=(12, 6))
plt.scatter(X_poly_train, y_poly_train, alpha=0.6, label='Training data', color='steelblue', edgecolors='k')

X_grid = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)

for degree, color in zip(degrees, colors):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_poly_train, y_poly_train)
    y_grid = pipe.predict(X_grid)
    train_score = pipe.score(X_poly_train, y_poly_train)
    test_score = pipe.score(X_poly_test, y_poly_test)
    plt.plot(X_grid, y_grid, color=color, linewidth=2,
             label=f'Degree {degree} (Train R²={train_score:.3f}, Test R²={test_score:.3f})')

plt.xlim(-3.5, 3.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Polynomial Regression: Underfitting vs Overfitting')
plt.legend(fontsize=9)
plt.show()

## 11. Regularization: Ridge (L2), Lasso (L1), ElasticNet

When we have many features or high-degree polynomials, coefficients can blow up. Regularization adds a penalty to keep them small.

| Method | Penalty | Effect |
|---|---|---|
| **Ridge** (L2) | $\alpha \sum \beta_j^2$ | Shrinks coefficients evenly, handles multicollinearity |
| **Lasso** (L1) | $\alpha \sum |\beta_j|$ | Can zero out features (feature selection) |
| **ElasticNet** | $\alpha (\rho \text{L1} + (1-\rho) \text{L2})$ | Balance between Ridge and Lasso |

**When to use which:**
- **Ridge**: When all features matter; for multicollinearity
- **Lasso**: When you suspect only a few features are important
- **ElasticNet**: When you have a large number of correlated features

In [ ]:
# Regularization path: how coefficients change with alpha
np.random.seed(42)
X_reg, y_reg = make_regression(n_samples=200, n_features=10, noise=10, random_state=42)

alphas = np.logspace(-2, 3, 50)

ridge_coefs = []
lasso_coefs = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_reg, y_reg)
    ridge_coefs.append(ridge.coef_)
    
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_reg, y_reg)
    lasso_coefs.append(lasso.coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

for i in range(ridge_coefs.shape[1]):
    axes[0].plot(alphas, ridge_coefs[:, i], alpha=0.8)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (log scale)')
axes[0].set_ylabel('Coefficient magnitude')
axes[0].set_title('Ridge (L2) Regularization Path')
axes[0].axvline(1.0, color='k', linestyle='--', alpha=0.4, label='alpha=1')

for i in range(lasso_coefs.shape[1]):
    axes[1].plot(alphas, lasso_coefs[:, i], alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('Coefficient magnitude')
axes[1].set_title('Lasso (L1) Regularization Path (features drop to 0)')
axes[1].axvline(1.0, color='k', linestyle='--', alpha=0.4, label='alpha=1')

plt.tight_layout()
plt.show()

In [ ]:
# Compare Ridge, Lasso, ElasticNet side-by-side
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Ridge (alpha=10)': Ridge(alpha=10.0),
    'Lasso (alpha=1)': Lasso(alpha=1.0, max_iter=10000),
    'Lasso (alpha=10)': Lasso(alpha=10.0, max_iter=10000),
    'ElasticNet (alpha=1)': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
}

results = []
for name, model in models.items():
    model.fit(X_train_r, y_train_r)
    train_r2 = model.score(X_train_r, y_train_r)
    test_r2 = model.score(X_test_r, y_test_r)
    n_nonzero = np.sum(np.abs(model.coef_) > 1e-6) if hasattr(model, 'coef_') else 'N/A'
    results.append({
        'Model': name,
        'Train R²': f'{train_r2:.4f}',
        'Test R²': f'{test_r2:.4f}',
        'Non-zero Coefs': n_nonzero
    })

pd.DataFrame(results)

## 12. Feature Scaling Importance for Regularized Models

Regularization penalizes the **magnitude** of coefficients. If features are on different scales (e.g., age 0-100 vs. salary 30k-200k), the penalty is applied unfairly — features with larger values get penalized more even if they're equally important.

**Always scale features before applying Ridge, Lasso, or ElasticNet.**

In [ ]:
# Demonstrate the impact of scaling on Lasso
np.random.seed(42)
n_samples = 200
X_unscaled = np.column_stack([
    np.random.randn(n_samples) * 100,   # feature 1: large scale
    np.random.randn(n_samples) * 0.01,  # feature 2: small scale
    np.random.randn(n_samples)          # feature 3: unit scale
])
y_scale = 3 * X_unscaled[:, 0] + 200 * X_unscaled[:, 1] + 5 * X_unscaled[:, 2] + np.random.normal(0, 10, n_samples)

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(X_unscaled, y_scale, test_size=0.3, random_state=42)

# Without scaling
lasso_no_scale = Lasso(alpha=1.0, max_iter=10000)
lasso_no_scale.fit(X_train_u, y_train_u)

# With scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_u)
X_test_scaled = scaler.transform(X_test_u)

lasso_scaled = Lasso(alpha=1.0, max_iter=10000)
lasso_scaled.fit(X_train_scaled, y_train_u)

print('Coefficients WITHOUT scaling:')
for i, c in enumerate(lasso_no_scale.coef_):
    print(f'  Feature {i+1}: {c:.4f}')
print(f'  Test R²: {lasso_no_scale.score(X_test_u, y_test_u):.4f}')
print(f'  Non-zero coefs: {np.sum(lasso_no_scale.coef_ != 0)}')
print()
print('Coefficients WITH scaling:')
for i, c in enumerate(lasso_scaled.coef_):
    print(f'  Feature {i+1}: {c:.4f}')
print(f'  Test R²: {lasso_scaled.score(X_test_scaled, y_test_u):.4f}')
print(f'  Non-zero coefs: {np.sum(lasso_scaled.coef_ != 0)}')

## 13. Cross-Validation with RidgeCV and LassoCV

Instead of manually tuning $\alpha$, use built-in cross-validation to find the optimal value automatically.

In [ ]:
alphas = np.logspace(-3, 3, 100)

ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_train_scaled, y_train_u)

lasso_cv = LassoCV(alphas=alphas, max_iter=10000, cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train_u)

print(f'RidgeCV optimal alpha: {ridge_cv.alpha_:.4f}')
print(f'RidgeCV Test R²: {ridge_cv.score(X_test_scaled, y_test_u):.4f}')
print()
print(f'LassoCV optimal alpha: {lasso_cv.alpha_:.4f}')
print(f'LassoCV Test R²: {lasso_cv.score(X_test_scaled, y_test_u):.4f}')
print(f'LassoCV non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)}')

## 14. Real-World Example: California Housing Dataset

We'll use the California housing dataset, which contains median house values for California districts based on features like income, housing age, and location.

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
X_housing = data.data
y_housing = data.target

print(f'Dataset shape: {df.shape}')
print(f'Features: {list(X_housing.columns)}')
print(f'Target: MedHouseVal (median house value in $100,000s)')
print()
df.head()

In [ ]:
# Quick exploratory visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), X_housing.columns):
    ax.scatter(X_housing[col], y_housing, alpha=0.3, s=5)
    ax.set_xlabel(col)
    ax.set_ylabel('MedHouseVal')
plt.tight_layout()
plt.show()

In [ ]:
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Build a pipeline with scaling + Ridge
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5))
])
pipeline.fit(X_h_train, y_h_train)

y_h_pred = pipeline.predict(X_h_test)

print('California Housing Results:')
print(f'  Optimal alpha: {pipeline.named_steps["ridge"].alpha_:.4f}')
print(f'  Test R²: {r2_score(y_h_test, y_h_pred):.4f}')
print(f'  Test RMSE: ${np.sqrt(mean_squared_error(y_h_test, y_h_pred)) * 100000:.0f}')
print(f'  Test MAE: ${mean_absolute_error(y_h_test, y_h_pred) * 100000:.0f}')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 8))
plt.scatter(y_h_test, y_h_pred, alpha=0.4, edgecolors='k')
plt.plot([y_h_test.min(), y_h_test.max()], [y_h_test.min(), y_h_test.max()], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Median House Value ($100k)')
plt.ylabel('Predicted Median House Value ($100k)')
plt.title('California Housing: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Learning Curves: Diagnosing Bias-Variance Tradeoff

Learning curves show how model performance changes with training set size. They help diagnose:
- **High bias (underfitting)**: Training and validation scores converge at a low value
- **High variance (overfitting)**: Large gap between training and validation scores

In [ ]:
def plot_learning_curve(model, X, y, title, cv=5, train_sizes=np.linspace(0.1, 1.0, 10)):
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X, y, train_sizes=train_sizes, cv=cv,
        scoring='r2', n_jobs=-1, random_state=42
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    plt.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.15, color='orange')
    plt.plot(train_sizes_abs, train_mean, 'o-', color='blue', label='Training score')
    plt.plot(train_sizes_abs, val_mean, 'o-', color='orange', label='Cross-validation score')
    plt.xlabel('Training examples')
    plt.ylabel('R² score')
    plt.title(f'Learning Curve: {title}')
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()

# Compare linear vs polynomial on housing data (sample for speed)
X_h_sample = X_housing.sample(500, random_state=42)
y_h_sample = y_housing.loc[X_h_sample.index]

scaler = StandardScaler()
X_h_scaled = scaler.fit_transform(X_h_sample)

plot_learning_curve(LinearRegression(), X_h_scaled, y_h_sample, 'Linear Regression')

poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])
plot_learning_curve(poly_pipe, X_h_scaled, y_h_sample, 'Polynomial Regression (degree=2)')

## 16. Model Interpretation: Coefficients and Feature Importance

One of the biggest advantages of linear regression is **interpretability**. Each coefficient tells us the expected change in the target for a one-unit change in the feature (after scaling for fair comparison).

In [ ]:
# Refit on full scaled data
scaler_full = StandardScaler()
X_h_train_s = scaler_full.fit_transform(X_h_train)
X_h_test_s = scaler_full.transform(X_h_test)

lr_final = LinearRegression()
lr_final.fit(X_h_train_s, y_h_train)

# Create coefficient DataFrame
coef_df = pd.DataFrame({
    'Feature': X_housing.columns,
    'Coefficient': lr_final.coef_,
    'Abs(Coefficient)': np.abs(lr_final.coef_)
}).sort_values('Abs(Coefficient)', ascending=False)

print('Feature Importance (standardized coefficients):')
print('Interpretation: "A one-standard-deviation increase in this feature changes'
      ' the median house value by (coefficient * $100,000)"')
print()
print(coef_df.to_string(index=False))
print(f'\nIntercept: {lr_final.intercept_:.4f} (baseline house value when all features are at their mean)')

In [ ]:
# Visualize coefficient importance
plt.figure(figsize=(10, 6))
colors = ['coral' if c < 0 else 'steelblue' for c in lr_final.coef_]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='k')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (scaled)')
plt.title('California Housing: Coefficient Importance')
plt.tight_layout()
plt.show()

print('Key insights:')
print('  - MedInc (median income) has the largest positive impact on house price')
print('  - HouseAge: older houses tend to be slightly more valuable in CA')
print('  - Latitude/Longitude capture geographic price variation')

## 17. Practice Exercises

Test your understanding with these exercises:

### Exercise 1: Implement Ridge Regression from Scratch
Modify the `LinearRegressionScratch` class to add L2 regularization. The closed-form solution becomes:
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$$

**Hint:** Add an `alpha` parameter and modify the `fit` method. Test that your implementation matches `sklearn.linear_model.Ridge`.

### Exercise 2: Forward Feature Selection
Write a function that performs forward feature selection: start with no features, iteratively add the feature that most improves the model's cross-validated R² score, and stop when adding a feature doesn't improve by more than 0.01.

### Exercise 3: Detect Multicollinearity with VIF
Write code to calculate the Variance Inflation Factor for each feature in the California housing dataset. Which features have VIF > 5 (indicating problematic multicollinearity)?

**Hint:** VIF for feature $j$ is $1/(1-R^2_j)$, where $R^2_j$ comes from regressing feature $j$ against all other features.

### Exercise 4: Compare Polynomial Degrees with Cross-Validation
For the synthetic polynomial dataset in Section 10, perform 5-fold cross-validation for degrees 1 through 12. Plot the mean CV R² (with error bars) against degree. Which degree would you choose? Explain why.

### Exercise 5: Robust Regression with Outliers
Add 10 outlier points to the California housing dataset (e.g., extreme MedHouseVal values with unusual feature combinations). Compare how LinearRegression, Ridge, and `sklearn.linear_model.HuberRegressor` handle these outliers. Which is most robust?

---

*Solutions are left as an exercise for the reader — but don't hesitate to revisit earlier sections for guidance!*